# Práctica — Clase 11. Arquitecturas de agentes: de A3C a IMPALA

**DIH DATalife — Curso de Reinforcement Learning (RL) desde cero**

En la clase viste "Catch": el banco de pruebas recomendado antes de complicar cualquier agente con paralelismo (A3C) o con arquitecturas distribuidas (IMPALA). En esta práctica trabajas con ese mismo entorno en dos ejercicios:

1. **Completas tú misma o tú mismo la función de paso (`step`) del entorno**, desde cero: cómo cae la pelota y cómo se mueve la pala
2. **Parametrizas el tamaño de la cuadrícula y la velocidad de caída**, y comparas cuántos episodios necesita el agente para aprender bien en cada configuración

Solo necesitas `numpy`, `torch` y `matplotlib`, las tres preinstaladas en Google Colab.

**Antes de empezar, ejecuta la celda siguiente para importar las librerías y fijar la semilla.**


In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

np.random.seed(0)
torch.manual_seed(0)


## Ejercicio 1 — Completa la función de paso de "Catch"

En la clase viste la clase `Catch` ya construida. Aquí tienes el mismo andamiaje, pero con el método `step` vacío: te toca completarlo tú.

Recuerda la mecánica del juego: la pelota aparece en una columna aleatoria de la fila superior y baja una fila por paso; controlas una pala de `ANCHO_PALA` píxeles de ancho en la fila inferior, que se mueve a la izquierda, a la derecha o se queda quieta. Cuando la pelota llega a la fila inferior, recibes +1 si la pala la cubre y -1 si no.

**Tu tarea es completar `step`, donde pone `# TODO`:**

1. Mueve la pala según la acción recibida (0 = izquierda, 1 = quieta, 2 = derecha), sin dejar que se salga de la cuadrícula
2. Haz que la pelota caiga una fila
3. Determina si el episodio termina (la pelota llegó a la última fila)
4. Si terminó, calcula la recompensa: +1.0 si la pala cubre la columna de la pelota, -1.0 si no. Si no terminó, la recompensa es 0.0
5. Devuelve el nuevo estado, la recompensa y si terminó

**Pistas**

- El diccionario `{0: -1, 1: 0, 2: 1}` traduce cada acción a un desplazamiento de columna
- Usa `np.clip(valor, minimo, maximo)` para no dejar que la pala se salga de la cuadrícula: el rango válido de `columna_pala` va de 0 a `COLUMNAS - ANCHO_PALA`
- El episodio termina cuando `self.fila_pelota == FILAS - 1`
- La pala cubre la pelota cuando `self.columna_pala <= self.columna_pelota < self.columna_pala + ANCHO_PALA`
- El estado siempre se calcula igual, ya está hecho en `_estado()`: no necesitas tocarlo

Con el `# TODO` sin completar, la celda de prueba de abajo falla: esa es la señal de que aún te falta terminar `step`.


In [ ]:
# --- Andamiaje del entorno "Catch": completa step() ---
FILAS = 10
COLUMNAS = 5
ANCHO_PALA = 2
N_ACCIONES = 3  # 0 = izquierda, 1 = quieta, 2 = derecha


class Catch:
    """Implementacion minima del juego de juguete "Catch"."""

    def __init__(self):
        self.reset()

    def reset(self):
        self.fila_pelota = 0
        self.columna_pelota = np.random.randint(COLUMNAS)
        # Punto de partida aleatorio de la pala, sin salirse del borde
        self.columna_pala = np.random.randint(COLUMNAS - ANCHO_PALA + 1)
        return self._estado()

    def _estado(self):
        """Codifica el estado como una imagen binaria aplanada: un uno
        donde esta la pelota y unos donde ocupa la pala. Ya esta completa,
        no hace falta que la toques."""
        grid = np.zeros((FILAS, COLUMNAS), dtype=np.float32)
        grid[self.fila_pelota, self.columna_pelota] = 1.0
        grid[FILAS - 1, self.columna_pala:self.columna_pala + ANCHO_PALA] = 1.0
        return grid.flatten()

    def step(self, accion):
        # TODO 1: mueve la pala segun `accion` (0=izquierda, 1=quieta, 2=derecha)
        #         usa np.clip para no salirte del rango [0, COLUMNAS - ANCHO_PALA]

        # TODO 2: haz que la pelota caiga una fila (self.fila_pelota)

        # TODO 3: determina si el episodio termina: la pelota llego a la
        #         ultima fila (FILAS - 1)

        # TODO 4: si termino, calcula la recompensa (+1.0 si la pala cubre
        #         la columna de la pelota, -1.0 si no). Si no termino,
        #         la recompensa es 0.0

        # TODO 5: devuelve self._estado(), la recompensa y si termino
        pass


**Ejecuta la celda siguiente para probar tu implementación.** Coloca la pelota y la pala en posiciones conocidas y comprueba que `step` se comporta como esperas: que la pala se mueve, que la pelota cae, que la recompensa solo aparece al final y que acierta o falla según corresponda.


In [ ]:
# --- Celda de prueba: no hace falta que la edites ---
entorno_prueba = Catch()

# Paso intermedio: la pelota todavia no llega abajo
entorno_prueba.fila_pelota = 0
entorno_prueba.columna_pelota = 2
entorno_prueba.columna_pala = 0
_, recompensa, terminado = entorno_prueba.step(1)  # 1 = quieta
assert entorno_prueba.fila_pelota == 1, "la pelota deberia caer una fila"
assert entorno_prueba.columna_pala == 0, "con accion 'quieta' la pala no deberia moverse"
assert recompensa == 0.0 and not terminado, "a mitad de la caida no hay recompensa"
print("Paso intermedio: OK")

# Ultimo paso, pala mal colocada: debe fallar
entorno_prueba.fila_pelota = FILAS - 2
entorno_prueba.columna_pelota = 4
entorno_prueba.columna_pala = 0
_, recompensa, terminado = entorno_prueba.step(1)
assert terminado, "en la ultima fila el episodio debe terminar"
assert recompensa == -1.0, "la pala no cubre la pelota: deberia fallar"
print("Fallo esperado: OK")

# Ultimo paso, pala bien colocada: debe atrapar
entorno_prueba.fila_pelota = FILAS - 2
entorno_prueba.columna_pelota = 4
entorno_prueba.columna_pala = 3
_, recompensa, terminado = entorno_prueba.step(1)
assert terminado and recompensa == 1.0, "la pala cubre la pelota: deberia atrapar"
print("Acierto esperado: OK")

# La pala no debe salirse de los bordes
entorno_prueba.columna_pala = 0
entorno_prueba.fila_pelota = 0
entorno_prueba.step(0)  # izquierda, ya en el borde
assert entorno_prueba.columna_pala == 0, "no deberia salirse del borde izquierdo"
entorno_prueba.columna_pala = COLUMNAS - ANCHO_PALA
entorno_prueba.step(2)  # derecha, ya en el borde
assert entorno_prueba.columna_pala == COLUMNAS - ANCHO_PALA, "no deberia salirse del borde derecho"
print("Bordes: OK")

print("\n¡Los cuatro controles pasan! Tu step() funciona correctamente.")


**Con `step` funcionando, entrena un agente de verdad para confirmarlo.** Reutilizamos el mismo actor-crítico de la clase, con menos episodios para que veas el resultado rápido.

**Ejecuta esta celda: si tu entorno funciona, la tasa de aciertos de los últimos 100 episodios debería quedar muy por encima de la de los primeros 100.**


In [ ]:
# --- Entrenamiento corto para confirmar que el entorno funciona de punta a punta ---
class ActorCritico(nn.Module):
    def __init__(self, dim_entrada, n_acciones, dim_oculta=128):
        super().__init__()
        self.base = nn.Sequential(nn.Linear(dim_entrada, dim_oculta), nn.ReLU())
        self.cabeza_politica = nn.Linear(dim_oculta, n_acciones)
        self.cabeza_valor = nn.Linear(dim_oculta, 1)

    def forward(self, x):
        h = self.base(x)
        return self.cabeza_politica(h), self.cabeza_valor(h)


GAMMA = 0.99
TASA_APRENDIZAJE = 1e-3
COEF_VALOR = 0.5
COEF_ENTROPIA = 0.01
N_EPISODIOS_PRUEBA = 1500

torch.manual_seed(0)
np.random.seed(0)
entorno = Catch()
red = ActorCritico(dim_entrada=FILAS * COLUMNAS, n_acciones=N_ACCIONES)
optimizador = torch.optim.Adam(red.parameters(), lr=TASA_APRENDIZAJE)


def ejecutar_episodio():
    estado = entorno.reset()
    log_probs, valores, recompensas, entropias = [], [], [], []
    terminado = False
    while not terminado:
        entrada = torch.from_numpy(estado)
        logits, valor = red(entrada)
        distribucion = torch.distributions.Categorical(logits=logits)
        accion = distribucion.sample()
        estado, recompensa, terminado = entorno.step(accion.item())
        log_probs.append(distribucion.log_prob(accion))
        entropias.append(distribucion.entropy())
        valores.append(valor.squeeze())
        recompensas.append(recompensa)
    return log_probs, valores, recompensas, entropias


def actualizar_red(log_probs, valores, recompensas, entropias):
    retornos = []
    G = 0.0
    for r in reversed(recompensas):
        G = r + GAMMA * G
        retornos.insert(0, G)
    retornos = torch.tensor(retornos, dtype=torch.float32)
    valores = torch.stack(valores)
    ventaja = retornos - valores.detach()
    perdida_actor = -(torch.stack(log_probs) * ventaja).mean()
    perdida_critico = F.mse_loss(valores, retornos)
    perdida_entropia = torch.stack(entropias).mean()
    perdida = perdida_actor + COEF_VALOR * perdida_critico - COEF_ENTROPIA * perdida_entropia
    optimizador.zero_grad()
    perdida.backward()
    optimizador.step()


aciertos = []
for episodio in range(N_EPISODIOS_PRUEBA):
    log_probs, valores, recompensas, entropias = ejecutar_episodio()
    actualizar_red(log_probs, valores, recompensas, entropias)
    aciertos.append(1 if recompensas[-1] > 0 else 0)

print("Tasa de aciertos, primeros 100 episodios:", round(np.mean(aciertos[:100]), 2))
print("Tasa de aciertos, ultimos 100 episodios:", round(np.mean(aciertos[-100:]), 2))


<details>
<summary>💡 Ver solución</summary>

```python
class Catch:
    """Implementación mínima del juego de juguete "Catch"."""

    def __init__(self):
        self.reset()

    def reset(self):
        self.fila_pelota = 0
        self.columna_pelota = np.random.randint(COLUMNAS)
        self.columna_pala = np.random.randint(COLUMNAS - ANCHO_PALA + 1)
        return self._estado()

    def _estado(self):
        grid = np.zeros((FILAS, COLUMNAS), dtype=np.float32)
        grid[self.fila_pelota, self.columna_pelota] = 1.0
        grid[FILAS - 1, self.columna_pala:self.columna_pala + ANCHO_PALA] = 1.0
        return grid.flatten()

    def step(self, accion):
        # 1. Movemos la pala; np.clip evita que se salga de la cuadrícula
        desplazamiento = {0: -1, 1: 0, 2: 1}[accion]
        self.columna_pala = int(
            np.clip(self.columna_pala + desplazamiento, 0, COLUMNAS - ANCHO_PALA)
        )
        # 2. La pelota cae una fila en cada paso
        self.fila_pelota += 1
        # 3. El episodio termina cuando la pelota llega a la última fila
        terminado = self.fila_pelota == FILAS - 1
        # 4. Solo hay recompensa distinta de cero al terminar
        if terminado:
            atrapada = (
                self.columna_pala
                <= self.columna_pelota
                < self.columna_pala + ANCHO_PALA
            )
            recompensa = 1.0 if atrapada else -1.0
        else:
            recompensa = 0.0
        # 5. Devolvemos el nuevo estado, la recompensa y si terminó
        return self._estado(), recompensa, terminado
```
</details>


## Ejercicio 2 — Parametriza la dificultad y compara cuánto tarda el agente en aprender

En la clase viste que "Catch" usa constantes fijas: diez filas, cinco columnas, una pala de dos píxeles, una pelota que cae una fila por paso. Ahora vas a convertir esos valores en parámetros configurables, para poder comparar configuraciones distintas sin tocar el código del entorno cada vez.

**Tu tarea tiene dos partes, marcadas con `# TODO` en la celda siguiente:**

**Parte A — Parametriza el entorno.** Completa `CatchParametrizable` para que `filas`, `columnas`, `ancho_pala` y `velocidad_caida` (cuántas filas cae la pelota en cada paso) sean atributos configurables por el constructor, en vez de las constantes globales `FILAS`, `COLUMNAS` y `ANCHO_PALA` que usaste en el ejercicio uno.

**Parte B — Diseña el experimento.** Completa el bucle que entrena un agente con cada configuración de `EXPERIMENTOS` y guarda, para cada una, cuántos episodios tardó en alcanzar una tasa de aciertos de 0.8 (calculada con una media móvil de 100 episodios).

**Pistas**

- En el constructor, guarda cada argumento como atributo de instancia: `self.filas = filas`, y lo mismo con los otros tres
- Dentro de `reset`, `_estado` y `step`, sustituye cada referencia a `FILAS`, `COLUMNAS` y `ANCHO_PALA` por `self.filas`, `self.columnas` y `self.ancho_pala`
- La pelota ahora cae `self.velocidad_caida` filas por paso, no una sola; el episodio termina cuando `self.fila_pelota >= self.filas - 1`
- Para la parte B, usa la función `entrenar_agente(entorno, n_episodios)` que ya tienes lista más abajo: te devuelve la lista de aciertos (uno o cero) episodio a episodio
- `episodios_hasta_umbral(aciertos)` también está ya lista: te devuelve en qué episodio la media móvil llegó por primera vez a 0.8, o `None` si nunca llega
- `EXPERIMENTOS` es un diccionario de nombre -> argumentos del entorno; recórrelo con `.items()`


In [ ]:
# --- Parte A: completa la parametrizacion del entorno ---
N_ACCIONES = 3  # 0 = izquierda, 1 = quieta, 2 = derecha


class CatchParametrizable:
    """Version de "Catch" con tamano de cuadricula, ancho de pala y
    velocidad de caida configurables."""

    def __init__(self, filas=10, columnas=5, ancho_pala=2, velocidad_caida=1):
        # TODO: guarda los cuatro argumentos como atributos de instancia
        # (self.filas, self.columnas, self.ancho_pala, self.velocidad_caida)
        self.reset()

    def reset(self):
        self.fila_pelota = 0
        # TODO: usa self.columnas y self.ancho_pala en vez de las
        # constantes globales para elegir columna_pelota y columna_pala
        return self._estado()

    def _estado(self):
        # TODO: construye la grid con np.zeros usando self.filas y
        # self.columnas, marca la posicion de la pelota y la de la pala
        # (igual que en el ejercicio uno, pero con los atributos de instancia)
        pass

    def step(self, accion):
        # TODO: mueve la pala con self.columnas y self.ancho_pala
        # TODO: haz que la pelota caiga self.velocidad_caida filas
        # TODO: el episodio termina cuando self.fila_pelota >= self.filas - 1
        # TODO: calcula la recompensa igual que en el ejercicio uno
        # TODO: devuelve self._estado(), la recompensa y si termino
        pass


**Ejecuta la celda siguiente para comprobar la parte A.** Crea dos entornos con tamaños distintos y revisa que el estado que devuelven tiene el tamaño correcto en cada caso.


In [ ]:
# --- Celda de prueba de la parte A: no hace falta que la edites ---
entorno_5x10 = CatchParametrizable(filas=10, columnas=5, ancho_pala=2, velocidad_caida=1)
entorno_8x10 = CatchParametrizable(filas=10, columnas=8, ancho_pala=2, velocidad_caida=1)

estado_5x10 = entorno_5x10.reset()
estado_8x10 = entorno_8x10.reset()

assert estado_5x10.shape[0] == 10 * 5, "el estado deberia tener 10*5 posiciones"
assert estado_8x10.shape[0] == 10 * 8, "el estado deberia tener 10*8 posiciones"
print("Tamano de estado por configuracion: OK")

estado, recompensa, terminado = entorno_5x10.step(1)
assert not terminado, "con velocidad_caida=1 en fila 0 no deberia terminar todavia"
print("Un paso de step(): OK")

print("\n¡La parametrizacion funciona! Sigue con la parte B.")


In [ ]:
# --- Actor-critico y funciones de entrenamiento reutilizables (ya completas) ---
GAMMA = 0.99
TASA_APRENDIZAJE = 1e-3
COEF_VALOR = 0.5
COEF_ENTROPIA = 0.01


class ActorCritico(nn.Module):
    def __init__(self, dim_entrada, n_acciones, dim_oculta=128):
        super().__init__()
        self.base = nn.Sequential(nn.Linear(dim_entrada, dim_oculta), nn.ReLU())
        self.cabeza_politica = nn.Linear(dim_oculta, n_acciones)
        self.cabeza_valor = nn.Linear(dim_oculta, 1)

    def forward(self, x):
        h = self.base(x)
        return self.cabeza_politica(h), self.cabeza_valor(h)


def ejecutar_episodio(entorno, red):
    estado = entorno.reset()
    log_probs, valores, recompensas, entropias = [], [], [], []
    terminado = False
    while not terminado:
        entrada = torch.from_numpy(estado)
        logits, valor = red(entrada)
        distribucion = torch.distributions.Categorical(logits=logits)
        accion = distribucion.sample()
        estado, recompensa, terminado = entorno.step(accion.item())
        log_probs.append(distribucion.log_prob(accion))
        entropias.append(distribucion.entropy())
        valores.append(valor.squeeze())
        recompensas.append(recompensa)
    return log_probs, valores, recompensas, entropias


def actualizar_red(red, optimizador, log_probs, valores, recompensas, entropias):
    retornos = []
    G = 0.0
    for r in reversed(recompensas):
        G = r + GAMMA * G
        retornos.insert(0, G)
    retornos = torch.tensor(retornos, dtype=torch.float32)
    valores = torch.stack(valores)
    ventaja = retornos - valores.detach()
    perdida_actor = -(torch.stack(log_probs) * ventaja).mean()
    perdida_critico = F.mse_loss(valores, retornos)
    perdida_entropia = torch.stack(entropias).mean()
    perdida = perdida_actor + COEF_VALOR * perdida_critico - COEF_ENTROPIA * perdida_entropia
    optimizador.zero_grad()
    perdida.backward()
    optimizador.step()


def entrenar_agente(entorno, n_episodios, semilla=0):
    """Entrena un actor-critico desde cero sobre `entorno` y devuelve la
    lista de aciertos (1 o 0) episodio a episodio."""
    torch.manual_seed(semilla)
    np.random.seed(semilla)
    dim_entrada = entorno.filas * entorno.columnas
    red = ActorCritico(dim_entrada=dim_entrada, n_acciones=N_ACCIONES)
    optimizador = torch.optim.Adam(red.parameters(), lr=TASA_APRENDIZAJE)
    aciertos = []
    for _ in range(n_episodios):
        log_probs, valores, recompensas, entropias = ejecutar_episodio(entorno, red)
        actualizar_red(red, optimizador, log_probs, valores, recompensas, entropias)
        aciertos.append(1 if recompensas[-1] > 0 else 0)
    return aciertos


def episodios_hasta_umbral(aciertos, ventana=100, umbral=0.8):
    """Devuelve el primer episodio en el que la media movil de aciertos
    alcanza `umbral`, o None si nunca lo alcanza."""
    if len(aciertos) < ventana:
        return None
    tasa_movil = np.convolve(aciertos, np.ones(ventana) / ventana, mode="valid")
    indices = np.where(tasa_movil >= umbral)[0]
    if len(indices) == 0:
        return None
    return int(indices[0] + ventana)


**Parte B.** Completa el bucle experimental de la celda siguiente: recorre `EXPERIMENTOS`, entrena un agente con cada configuración y guarda en `resultados` cuántos episodios tardó en llegar al 80% de aciertos. Ya tienes dos configuraciones de ejemplo en `EXPERIMENTOS`; puedes añadir las tuyas propias cambiando `columnas`, `ancho_pala` o `velocidad_caida`.

**Aviso:** esta celda entrena varios agentes seguidos y puede tardar uno o dos minutos en Colab.


In [ ]:
# --- Parte B: completa el bucle experimental ---
N_EPISODIOS = 2000

EXPERIMENTOS = {
    "base (10x5, pala=2, vel=1)": dict(filas=10, columnas=5, ancho_pala=2, velocidad_caida=1),
    "mas columnas (10x8, pala=2, vel=1)": dict(filas=10, columnas=8, ancho_pala=2, velocidad_caida=1),
    # TODO: anade aqui una tercera configuracion propia, por ejemplo
    # cambiando ancho_pala a 1 o velocidad_caida a 2
}

resultados = {}

# TODO: recorre EXPERIMENTOS con .items(), crea un CatchParametrizable con
# cada configuracion (usa **argumentos), entrena con entrenar_agente y
# guarda en resultados[nombre] una tupla (aciertos, episodios_hasta_umbral)


In [ ]:
# --- Graficamos la comparacion (ya completa, no hace falta tocarla) ---
plt.figure(figsize=(9, 5))
for nombre, (aciertos, ep_umbral) in resultados.items():
    ventana = 100
    tasa_movil = np.convolve(aciertos, np.ones(ventana) / ventana, mode="valid")
    etiqueta = f"{nombre} — 80% en episodio {ep_umbral}" if ep_umbral else f"{nombre} — no llega al 80%"
    plt.plot(tasa_movil, label=etiqueta)

plt.xlabel("Episodios")
plt.ylabel("Tasa de aciertos (media movil de 100 episodios)")
plt.title('Efecto del tamano de cuadricula y la velocidad de caida en "Catch"')
plt.axhline(0.8, color="gray", linestyle="--", alpha=0.5)
plt.ylim(0, 1)
plt.legend()
plt.grid(alpha=0.3)
plt.show()

for nombre, (_, ep_umbral) in resultados.items():
    print(nombre, "-> episodios hasta el 80% de aciertos:", ep_umbral)


<details>
<summary>💡 Ver solución</summary>

```python
# --- Parte A: entorno parametrizado ---
class CatchParametrizable:
    # Version de "Catch" con tamano de cuadricula, ancho de pala y
    # velocidad de caida configurables.

    def __init__(self, filas=10, columnas=5, ancho_pala=2, velocidad_caida=1):
        self.filas = filas
        self.columnas = columnas
        self.ancho_pala = ancho_pala
        self.velocidad_caida = velocidad_caida
        self.reset()

    def reset(self):
        self.fila_pelota = 0
        self.columna_pelota = np.random.randint(self.columnas)
        self.columna_pala = np.random.randint(self.columnas - self.ancho_pala + 1)
        return self._estado()

    def _estado(self):
        grid = np.zeros((self.filas, self.columnas), dtype=np.float32)
        fila_visible = min(self.fila_pelota, self.filas - 1)
        grid[fila_visible, self.columna_pelota] = 1.0
        grid[self.filas - 1, self.columna_pala:self.columna_pala + self.ancho_pala] = 1.0
        return grid.flatten()

    def step(self, accion):
        desplazamiento = {0: -1, 1: 0, 2: 1}[accion]
        self.columna_pala = int(
            np.clip(self.columna_pala + desplazamiento, 0, self.columnas - self.ancho_pala)
        )
        self.fila_pelota += self.velocidad_caida
        terminado = self.fila_pelota >= self.filas - 1
        if terminado:
            atrapada = (
                self.columna_pala
                <= self.columna_pelota
                < self.columna_pala + self.ancho_pala
            )
            recompensa = 1.0 if atrapada else -1.0
        else:
            recompensa = 0.0
        return self._estado(), recompensa, terminado


# --- Parte B: bucle experimental ---
N_EPISODIOS = 2000

EXPERIMENTOS = {
    "base (10x5, pala=2, vel=1)": dict(filas=10, columnas=5, ancho_pala=2, velocidad_caida=1),
    "mas columnas (10x8, pala=2, vel=1)": dict(filas=10, columnas=8, ancho_pala=2, velocidad_caida=1),
    "pala mas estrecha (10x5, pala=1, vel=1)": dict(filas=10, columnas=5, ancho_pala=1, velocidad_caida=1),
}

resultados = {}
for nombre, config in EXPERIMENTOS.items():
    entorno = CatchParametrizable(**config)
    aciertos = entrenar_agente(entorno, N_EPISODIOS)
    ep_umbral = episodios_hasta_umbral(aciertos)
    resultados[nombre] = (aciertos, ep_umbral)
```

Con estas tres configuraciones verás que la base (diez filas, cinco columnas, pala de dos píxeles) llega al 80% de aciertos en unos cientos de episodios, mientras que ampliar la cuadrícula a ocho columnas o reducir la pala a un solo píxel obliga al agente a tardar bastante más: hay más columnas posibles donde puede caer la pelota, o menos margen de error para cubrirla, así que la política necesita más experiencia para afinar la puntería.
</details>


## Para seguir practicando

**Sube `velocidad_caida` a 3 o 4 en una configuración nueva y vuelve a ejecutar el experimento.** ¿Tarda más o menos en llegar al 80% de aciertos que la configuración base? Piensa por qué: con una pelota que cae más rápido, el agente tiene menos pasos por episodio para corregir la posición de la pala, pero también cada episodio es más corto y más barato de entrenar.

Anota en qué configuración exacta empieza a fallar tu agente en 2.000 episodios y qué cambio —más episodios, más entropía, una red más grande— lo vuelve a arreglar. Ese ejercicio de tantear los límites de un problema pequeño es, palabra por palabra, el consejo práctico central de la clase 11.
